# 🌍 Data Handling & Extraction (Geospatial & Climate Data)

This notebook manually imports, processes, and cleanly structures the real-world datasets from WRI Aqueduct, FAO AQUASTAT, and NASA GRACE.

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='viridis')

## 1. WRI Aqueduct 4.0 (Geospatial Water Risk)

In [ ]:
# Load Aqueduct
import warnings
warnings.filterwarnings('ignore')

aqua_path = "data/raw/aqueduct/Aqueduct40_waterrisk_download_Y2023M07D05/CVS/Aqueduct40_baseline_annual_y2023m07d05.csv"
if os.path.exists(aqua_path):
    print("Loading Aqueduct...")
    df_aq = pd.read_csv(aqua_path, low_memory=False)
    print(f"Raw shape: {df_aq.shape}")
    
    # Aggregation
    df_aq = df_aq.replace(9999, np.nan)
    cols_to_keep = [c for c in df_aq.columns if "_score" in c or "_raw" in c or "_cat" in c or "_label" in c]
    df_country = df_aq.groupby(["name_0", "iso_a3"])[cols_to_keep].mean(numeric_only=True).reset_index()
    print(f"Processed shape: {df_country.shape}")
    
    display(df_country[['name_0', 'bws_score', 'drr_score', 'gtd_score']].head())
else:
    print(f"Raw file not found: {aqua_path}")

## 2. FAO AQUASTAT (Water Resources)

In [ ]:
# Load AQUASTAT
stat_path = "data/raw/aquastat/bulk_eng(in).csv"
if os.path.exists(stat_path):
    print("Loading AQUASTAT...")
    df_stat = pd.read_csv(stat_path, encoding="latin-1")
    print(f"Raw shape: {df_stat.shape}")
    # Quick pivot
    df_stat.columns = ["var_id", "variable", "area_id", "country", "year_id", "year", "value", "symb", "desc"]
    df_stat = df_stat[df_stat["variable"].str.contains("Total renewable water resources|SDG 6.4.2|Precipitation", case=False, na=False)]
    
    pivot = df_stat.pivot_table(index=["country", "year"], columns="variable", values="value").reset_index()
    print(f"Processed pivot shape: {pivot.shape}")
    display(pivot.head())
else:
    print(f"Raw file not found: {stat_path}")

## 3. NASA GRACE (Geospatial Satellite Anomaly Data)

In [ ]:
grace_path = "data/raw/grace/1.nc"
if os.path.exists(grace_path):
    ds = xr.open_dataset(grace_path)
    print(ds)
    # Plot global mean TWS anomaly over time
    if 'lwe_thickness' in ds.variables:
        tws_mean = ds['lwe_thickness'].mean(dim=['lat', 'lon'])
        tws_mean.plot(figsize=(10,4))
        plt.title('Global Mean Terrestrial Water Storage Anomaly (cm)')
        plt.show()
else:
    print(f"Raw file not found: {grace_path}")